# 考古器物计数特征提取 Pipeline
## 东汉 → 魏晋十六国 → 北魏 朝代识别

**Created by:** qodercli  
**Date:** 2026-07-12  
**Data source:** `./data/`

---

### 目标
从考古学 markdown 文献中动态提取计数特征（quantitative features），构建朝代识别模型。

### 数据来源
- 山东地区汉代墓葬的考古学研究（刘剑）
- 幽燕地区两汉墓葬研究
- 光宅中原：拓跋至北魏的墓葬文化
- 京津冀汉墓研究
- 图像与装饰：北朝墓葬
- 各类发掘报告（屯留后河、朝阳袁台子、长清西汉济北王陵等）
- 专题论文（无字印研究等）

## Pipeline 架构总览

```
Markdown 文件
    │
    ├── Layer 1: 预处理 + 朝代感知分块
    │
    ├── Layer 2: LLM 动态特征提取（两阶段）
    │     ├── 阶段 A: 特征发现 (Schema Discovery)
    │     └── 阶段 B: 特征量化 (Structured Extraction)
    │
    ├── Layer 3: 朝代判别力分析
    │
    └── Layer 4: 验证 + 迭代
```

---
## Layer 1: 预处理 + 朝代感知分块

### 输入处理
1. **表格提取** → HTML table → 结构化 DataFrame (`pandas.read_html` / `BeautifulSoup`)
2. **叙述文本** → 按朝代关键词分段
3. **元数据标注** → 每个 chunk 附带来源、时期、区域、器物类别

### 分块策略
**不做固定大小 chunk**，而是按**语义段落 + 朝代边界**切分：
- 触发词: `东汉` / `西汉` / `魏晋` / `十六国` / `北魏` / `拓跋` / `平城` / `洛阳` / `大同`
- 一个 chunk 可能是一张表，也可能是一段分析文字

### 输出格式
```python
{
    "chunk_id": "file_001_chunk_003",
    "source_file": "wuziyin.md",
    "inferred_dynasty": "东汉",
    "region": "全国",
    "artifact_category": "印章",
    "content_type": "table",  # or "narrative"
    "text": "...",
    "table_data": [...]  # if table
}
```

---
## Layer 2: LLM 动态特征提取（核心）

### 阶段 A — 特征发现 (Schema Discovery)

不预设 schema，让 LLM 从数据中**自然发现**特征：

```python
prompt = f"""
从以下考古学文本中提取所有可量化的器物/墓葬特征。
不要预设类别，让特征自然浮现。

输出格式：
- feature_name: 特征名
- feature_type: numeric | categorical | boolean | ratio
- dynasty_signal: 该特征对哪个时期有区分度
- evidence: 原文依据

文本：{chunk}
"""
```

### 阶段 B — 特征量化 (Structured Extraction)

用阶段 A 发现的特征词典，对每个 chunk 做结构化提取：

```python
prompt = f"""
根据以下特征清单，从文本中提取具体数值/类别。

特征清单：{discovered_features}
文本：{chunk}

输出 JSON：
{{
  "dynasty": "东汉",
  "region": "山东",
  "features": {{
    "无字印总数": 82,
    "玉印数量": 31,
    "坛钮占比": 0.41,
    ...
  }}
}}
"""
```

### 示例：动态发现的特征（预期）

| 特征名 | 类型 | 朝代信号 |
|--------|------|----------|
| 无字印材质分布 | categorical | 西汉>东汉 |
| 坛钮vs桥钮比例 | ratio | 西汉→东汉变化 |
| 墓室面积(㎡) | numeric | 等级指标 |
| 陶器组合类型 | categorical | 分期核心依据 |
| 砖室vs土洞墓 | categorical | 东汉→魏晋转变 |
| 壁画墓比例 | ratio | 东汉特征 |
| 随葬印章数量 | numeric | 西汉>东汉 |
| 墓葬形制（土坑/砖室/石室） | categorical | 时期转变标志 |
| 随葬品组合（鼎盒壶等） | categorical | 礼制变化 |

---
## Layer 3: 朝代判别力分析

```
所有提取结果 → 按时期聚合
    │
    ├── 频率统计：每个特征在各时期的分布
    │     例: 坛钮 西汉85% → 东汉15% → 魏晋<5%
    │
    ├── 信息增益 / 卡方检验：哪些特征对朝代区分度最高
    │
    └── 输出：
          - 特征重要性排序
          - 各时期"指纹"（关键特征组合）
          - 过渡期的混合信号
```

---
## Layer 4: 验证 + 迭代

```
已知年代的墓葬 → 用提取的特征做交叉验证
    │
    ├── 如果分类准确率高 → 特征有效
    ├── 如果某些墓葬被误判 → 回看缺失的特征
    └── 补充新的特征 → 回到阶段A迭代
```

---
## 技术选型

| 环节 | 推荐工具 | 理由 |
|------|----------|------|
| 表格解析 | `pandas.read_html` + `BeautifulSoup` | HTML 表格结构规整 |
| 文本分块 | 自定义 splitter（按朝代关键词 + 章节标题） | 通用 chunker 会切断语义 |
| 特征发现 | Claude/GPT-4 + few-shot prompt | 需要理解考古学术语 |
| 结构化提取 | LLM function calling / JSON mode | 保证输出格式一致 |
| 存储 | SQLite + JSON 字段 | 轻量，特征 schema 灵活 |
| 分析 | pandas + scipy（卡方/信息增益） | 统计朝代区分度 |
| 可视化 | matplotlib / plotly | 特征分布对比 |

---
## 动态适应的关键设计

1. **特征词典不预设** — 阶段 A 让 LLM 从数据中发现特征，而不是手动定义
2. **增量更新** — 新文件进来后，先跑阶段 A 看有没有新特征，有则扩充词典
3. **置信度标注** — 每个提取值附带置信度（LLM 自评），低置信度的标记人工复核
4. **朝代标签模糊化** — 允许"东汉中晚期"、"魏晋过渡"这种模糊标签，不强制离散分类

---
## 实施步骤

1. 拿一两个文件（如无字印论文 + 光宅中原）跑一遍阶段 A
2. 查看 LLM 能发现什么特征
3. 根据结果调整 prompt 和分块策略
4. 扩展到全部文件
5. 做朝代判别力分析
6. 交叉验证

---
## 开始实现

### 环境准备

In [ ]:
# 依赖安装（首次运行）
# !pip install pandas beautifulsoup4 lxml openai anthropic scipy matplotlib plotly tqdm

In [ ]:
import os
import json
import re
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

# 数据路径
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

### Step 1: 文件扫描

In [ ]:
# 扫描所有 markdown 文件
md_files = list(DATA_DIR.rglob("*.md"))
print(f"找到 {len(md_files)} 个 markdown 文件:\n")
for f in md_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.relative_to(DATA_DIR):<70} {size_kb:>8.1f} KB")

### Step 2: 朝代关键词定义

In [ ]:
# 朝代/时期关键词（用于分块和标注）
DYNASTY_KEYWORDS = {
    "西汉早期": ["西汉早期", "西汉初", "汉初", "高祖", "惠帝", "吕后", "文景"],
    "西汉中期": ["西汉中期", "武帝", "昭帝", "宣帝", "元帝", "成帝"],
    "西汉晚期": ["西汉晚期", "西汉末", "哀帝", "平帝", "孺子婴", "新莽", "王莽"],
    "东汉早期": ["东汉早期", "东汉初", "光武帝", "明帝", "章帝"],
    "东汉中期": ["东汉中期", "和帝", "安帝", "顺帝", "冲帝", "质帝"],
    "东汉晚期": ["东汉晚期", "东汉末", "桓帝", "灵帝", "献帝"],
    "魏晋": ["魏晋", "曹魏", "西晋", "魏晋时期"],
    "十六国": ["十六国", "五胡十六国", "前秦", "后秦", "前燕", "后燕", "北燕", "前赵", "后赵"],
    "北魏": ["北魏", "拓跋", "平城", "大同", "洛阳时期"],
    "北朝": ["北朝", "东魏", "西魏", "北齐", "北周"],
}

# 区域关键词
REGION_KEYWORDS = {
    "山东": ["山东", "齐鲁", "齐地", "鲁地", "胶东", "鲁北", "鲁南"],
    "幽燕": ["幽燕", "幽州", "燕", "北京", "河北北部"],
    "关中": ["关中", "长安", "西安", "陕西"],
    "中原": ["中原", "河南", "洛阳"],
    "山西": ["山西", "并州", "晋", "大同", "平城"],
    "京津冀": ["京津冀", "北京", "天津", "河北"],
}

print("朝代关键词:")
for dynasty, kws in DYNASTY_KEYWORDS.items():
    print(f"  {dynasty}: {kws[:3]}...")
print(f"\n区域关键词:")
for region, kws in REGION_KEYWORDS.items():
    print(f"  {region}: {kws[:3]}...")

### Step 3: 朝代感知分块器

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    source_file: str
    content_type: str  # "table" | "narrative"
    text: str
    inferred_dynasty: List[str] = field(default_factory=list)
    inferred_region: List[str] = field(default_factory=list)
    artifact_category: str = ""
    table_data: Optional[List[Dict]] = None


def detect_dynasty(text: str) -> List[str]:
    """从文本中检测朝代标签"""
    detected = []
    for dynasty, keywords in DYNASTY_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                detected.append(dynasty)
                break
    return detected


def detect_region(text: str) -> List[str]:
    """从文本中检测区域标签"""
    detected = []
    for region, keywords in REGION_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                detected.append(region)
                break
    return detected


def extract_tables(html_content: str) -> List[Dict]:
    """从 HTML 内容中提取表格"""
    soup = BeautifulSoup(html_content, "lxml")
    tables = []
    for table in soup.find_all("table"):
        rows = []
        for tr in table.find_all("tr"):
            cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            if cells:
                rows.append(cells)
        if rows:
            tables.append({"headers": rows[0], "data": rows[1:]})
    return tables


def chunk_file(filepath: Path) -> List[Chunk]:
    """将一个 markdown 文件切分为语义 chunk"""
    content = filepath.read_text(encoding="utf-8")
    chunks = []
    
    # 1. 提取表格作为独立 chunk
    tables = extract_tables(content)
    for i, table in enumerate(tables):
        table_text = str(table)
        chunk = Chunk(
            chunk_id=f"{filepath.stem}_table_{i}",
            source_file=filepath.name,
            content_type="table",
            text=table_text[:2000],  # 截断过长表格
            inferred_dynasty=detect_dynasty(table_text),
            inferred_region=detect_region(table_text),
            table_data=table,
        )
        chunks.append(chunk)
    
    # 2. 按章节标题切分叙述文本
    # 移除表格部分后处理
    text_only = re.sub(r"<table.*?</table>", "", content, flags=re.DOTALL)
    text_only = re.sub(r"<div.*?</div>", "", text_only, flags=re.DOTALL)
    
    # 按 markdown 标题分段
    sections = re.split(r"\n(?=#{1,4}\s)", text_only)
    for i, section in enumerate(sections):
        if len(section.strip()) < 50:  # 跳过过短的段
            continue
        chunk = Chunk(
            chunk_id=f"{filepath.stem}_section_{i}",
            source_file=filepath.name,
            content_type="narrative",
            text=section.strip()[:3000],
            inferred_dynasty=detect_dynasty(section),
            inferred_region=detect_region(section),
        )
        chunks.append(chunk)
    
    return chunks


print("分块器定义完成。")

### Step 4: 试运行 — 扫描一个文件

In [ ]:
# 先用无字印论文测试
test_file = DATA_DIR / "wuziyin.md"
if test_file.exists():
    test_chunks = chunk_file(test_file)
    print(f"文件: {test_file.name}")
    print(f"切分为 {len(test_chunks)} 个 chunk\n")
    
    for i, chunk in enumerate(test_chunks[:5]):  # 只看前5个
        print(f"--- Chunk {i} ---")
        print(f"  ID: {chunk.chunk_id}")
        print(f"  Type: {chunk.content_type}")
        print(f"  Dynasty: {chunk.inferred_dynasty}")
        print(f"  Region: {chunk.inferred_region}")
        print(f"  Text preview: {chunk.text[:150]}...")
        print()
else:
    print(f"文件不存在: {test_file}")

### Step 5: LLM 特征提取接口

下面定义 LLM 调用接口。支持 OpenAI 和 Anthropic。

In [ ]:
# LLM 调用封装
# 根据你的环境选择 backend

LLM_BACKEND = "openai"  # or "anthropic"

def call_llm(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    """统一的 LLM 调用接口"""
    if LLM_BACKEND == "openai":
        from openai import OpenAI
        client = OpenAI()
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=temperature,
        )
        return resp.choices[0].message.content
    
    elif LLM_BACKEND == "anthropic":
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            system=system or "你是一位考古学专家。",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
        )
        return resp.content[0].text
    
    else:
        raise ValueError(f"Unknown backend: {LLM_BACKEND}")


print(f"LLM backend: {LLM_BACKEND}")
print("接口定义完成。")

### Step 6: 阶段 A — 特征发现 Prompt

In [ ]:
SYSTEM_PROMPT_DISCOVERY = """你是一位精通中国考古学的专家， specializing in 汉代至北朝时期。
你的任务是从考古学文献中识别和提取可量化的器物/墓葬特征（计数特征）。
这些特征将用于区分不同历史时期（东汉、魏晋十六国、北魏等）。

要求：
1. 不要预设类别，让特征从文本中自然浮现
2. 关注可量化、可比较的特征
3. 标注每个特征的朝代区分度
4. 给出原文依据"""

DISCOVERY_PROMPT_TEMPLATE = """请从以下考古学文本中提取所有可量化的器物/墓葬特征。

文本类型: {content_type}
推断时期: {inferred_dynasty}
推断区域: {inferred_region}

文本内容:
---
{text}
---

请输出 JSON 格式：
{{
  "features": [
    {{
      "feature_name": "特征名称",
      "feature_type": "numeric|categorical|boolean|ratio",
      "value": "具体值（如果是表格数据）",
      "dynasty_signal": "该特征对哪些时期有区分度",
      "evidence": "原文中的依据（引用）",
      "confidence": 0.0-1.0
    }}
  ]
}}

只输出 JSON，不要其他内容。"""


def discover_features(chunk: Chunk) -> Dict:
    """阶段 A: 从单个 chunk 发现特征"""
    prompt = DISCOVERY_PROMPT_TEMPLATE.format(
        content_type=chunk.content_type,
        inferred_dynasty=", ".join(chunk.inferred_dynasty) or "未明确",
        inferred_region=", ".join(chunk.inferred_region) or "未明确",
        text=chunk.text,
    )
    
    response = call_llm(prompt, system=SYSTEM_PROMPT_DISCOVERY)
    
    # 解析 JSON
    try:
        # 尝试提取 JSON 块
        json_match = re.search(r"\{[\s\S]*\}", response)
        if json_match:
            return json.loads(json_match.group())
        return {"features": [], "raw_response": response}
    except json.JSONDecodeError:
        return {"features": [], "raw_response": response, "parse_error": True}


print("特征发现 prompt 定义完成。")

### Step 7: 试运行 — 对一个 chunk 做特征发现

In [ ]:
# 选一个有内容的 chunk 测试
if test_chunks:
    # 找一个叙述类型的 chunk
    narrative_chunks = [c for c in test_chunks if c.content_type == "narrative" and len(c.text) > 200]
    if narrative_chunks:
        sample_chunk = narrative_chunks[0]
        print(f"测试 chunk: {sample_chunk.chunk_id}")
        print(f"时期: {sample_chunk.inferred_dynasty}")
        print(f"文本预览: {sample_chunk.text[:200]}...\n")
        
        # 调用 LLM
        # result = discover_features(sample_chunk)
        # print(json.dumps(result, ensure_ascii=False, indent=2))
        
        print("取消注释上面的代码来运行 LLM 调用。")
        print("（需要配置 API key）")
    else:
        print("没有找到合适的叙述类型 chunk")

### Step 8: 批量处理 + 特征词典聚合

In [ ]:
def process_all_files(file_pattern: str = "*.md") -> List[Chunk]:
    """处理所有文件，返回所有 chunks"""
    all_chunks = []
    files = list(DATA_DIR.rglob(file_pattern))
    
    for filepath in tqdm(files, desc="切分文件"):
        chunks = chunk_file(filepath)
        all_chunks.extend(chunks)
    
    print(f"\n总计: {len(all_chunks)} 个 chunks")
    return all_chunks


def aggregate_feature_vocab(all_features: List[Dict]) -> pd.DataFrame:
    """聚合所有发现的特征，生成特征词典"""
    feature_list = []
    for result in all_features:
        for f in result.get("features", []):
            feature_list.append(f)
    
    df = pd.DataFrame(feature_list)
    if not df.empty:
        # 去重 + 统计出现频率
        vocab = df.groupby("feature_name").agg({
            "feature_type": "first",
            "dynasty_signal": lambda x: ", ".join(set(", ".join(x).split(", "))),
            "evidence": "count",  # 出现次数
        }).rename(columns={"evidence": "occurrence_count"})
        return vocab.sort_values("occurrence_count", ascending=False)
    return df


print("批处理函数定义完成。")

---
## 下一步

1. 配置 API key（OpenAI 或 Anthropic）
2. 运行特征发现（阶段 A）
3. 审查发现的特征，调整 prompt
4. 运行特征量化（阶段 B）
5. 朝代判别力分析